## SETUP

In [2]:
from pathlib import Path
import duckdb
import os
import time

PROJECT_ROOT = Path(r"C:\Users\USUARIO WINDOWS\CODIGOS\PAZ CD").resolve()
os.chdir(PROJECT_ROOT)

DB_PATH = PROJECT_ROOT / "data" / "exoplanets.duckdb"
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
ART_DIR = PROJECT_ROOT / "artifacts"
DOCS_DIR = PROJECT_ROOT / "docs"

ART_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

if not RAW_CSV.exists():
    raise FileNotFoundError(f"No encuentro {RAW_CSV}")

con = duckdb.connect(str(DB_PATH))

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'", "''") + "'"

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"""
CREATE VIEW raw_ps AS
SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})
""")

print("Setup OK")

Setup OK


## RECONSTRUIR BASE MINIMA

In [14]:
con.execute("DROP TABLE IF EXISTS silver_planet_v3")

con.execute("""
CREATE TABLE silver_planet_v3 AS
SELECT
  pl_name,
  hostname,
  LOWER(TRIM(hostname)) AS hostname_canon,
  discoverymethod,
  LOWER(TRIM(discoverymethod)) AS discoverymethod_canon,
  TRY_CAST(disc_year AS INTEGER) AS disc_year_int,
  CASE
    WHEN TRY_CAST(disc_year AS INTEGER) IS NULL THEN TRUE
    WHEN TRY_CAST(disc_year AS INTEGER) < 1980 THEN TRUE
    WHEN TRY_CAST(disc_year AS INTEGER) > 2026 THEN TRUE
    ELSE FALSE
  END AS disc_year_bad,
  pl_orbper,
  pl_rade,
  pl_bmasse,
  pl_eqt,
  sy_dist,
  ra,
  dec,
  st_teff,
  st_rad,
  st_mass
FROM raw_ps
WHERE pl_name IS NOT NULL
  AND hostname IS NOT NULL
  AND (pl_rade IS NULL OR pl_rade > 0)
  AND (pl_bmasse IS NULL OR pl_bmasse > 0)
""")

con.sql("""
SELECT
  COUNT(*) AS n_rows,
  COUNT(DISTINCT pl_name) AS n_planets,
  COUNT(DISTINCT hostname_canon) AS n_hosts
FROM silver_planet_v3
""").show()

┌────────┬───────────┬─────────┐
│ n_rows │ n_planets │ n_hosts │
│ int64  │   int64   │  int64  │
├────────┼───────────┼─────────┤
│   6291 │      6291 │    4709 │
└────────┴───────────┴─────────┘



## FUNCION PARA MEDIR EL TIEMPO

In [15]:
def timed_query(sql: str, repeats: int = 3):
    times = []
    result = None

    for _ in range(repeats):
        start = time.perf_counter()
        result = con.sql(sql).fetchall()
        end = time.perf_counter()
        times.append(end - start)

    return {
        "min_s": min(times),
        "avg_s": sum(times) / len(times),
        "max_s": max(times),
        "rows": len(result),
        "result": result
    }

## CRITICA 1- BASELINE

In [16]:
q1_baseline = """
WITH wide AS (
  SELECT *
  FROM silver_planet_v3
  WHERE disc_year_int >= 2010
)
SELECT
  discoverymethod_canon,
  COUNT(*) AS n_planets,
  ROUND(AVG(pl_rade), 2) AS avg_radius,
  ROUND(AVG(pl_bmasse), 2) AS avg_mass
FROM wide
WHERE pl_rade IS NOT NULL
GROUP BY discoverymethod_canon
ORDER BY n_planets DESC
"""

q1_base_time = timed_query(q1_baseline)
q1_base_time

{'min_s': 0.004908599999907892,
 'avg_s': 0.006519199998971696,
 'max_s': 0.008315299994137604,
 'rows': 11,
 'result': [('transit', 4588, 4.23, 112.95),
  ('radial velocity', 841, 8.88, 967.02),
  ('microlensing', 268, 10.04, 835.53),
  ('imaging', 76, 15.32, 4492.27),
  ('transit timing variations', 40, 6.46, 483.47),
  ('eclipse timing variations', 14, 12.91, 2092.93),
  ('orbital brightness modulation', 6, 9.65, 350.32),
  ('astrometry', 6, 12.45, 4673.61),
  ('pulsar timing', 2, 7.56, 127.31),
  ('pulsation timing variations', 1, 12.4, 3750.39),
  ('disk kinematics', 1, 13.3, 794.58)]}

guardando el ` explain analize`

In [17]:
explain_q1 = con.sql("EXPLAIN ANALYZE " + q1_baseline).fetchall()
explain_q1_text = "\n".join(r[1] for r in explain_q1)

out_q1 = ART_DIR / "w11_explain_critical_q1.txt"
out_q1.write_text(explain_q1_text, encoding="utf-8")

print(explain_q1_text)
print("Guardado en:", out_q1)

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE  WITH wide AS (   SELECT *   FROM silver_planet_v3   WHERE disc_year_int >= 2010 ) SELECT   discoverymethod_canon,   COUNT(*) AS n_planets,   ROUND(AVG(pl_rade), 2) AS avg_radius,   ROUND(AVG(pl_bmasse), 2) AS avg_mass FROM wide WHERE pl_rade IS NOT NULL GROUP BY discoverymethod_canon ORDER BY n_planets DESC 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0055s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│           0 rows          │
│       

### reescritura query critica 1

In [18]:
q1_rewrite = """
SELECT
  discoverymethod_canon,
  COUNT(*) AS n_planets,
  ROUND(AVG(pl_rade), 2) AS avg_radius,
  ROUND(AVG(pl_bmasse), 2) AS avg_mass
FROM silver_planet_v3
WHERE disc_year_int >= 2010
  AND pl_rade IS NOT NULL
GROUP BY discoverymethod_canon
ORDER BY n_planets DESC
"""

q1_rewrite_time = timed_query(q1_rewrite)
q1_rewrite_time

{'min_s': 0.004960199999914039,
 'avg_s': 0.005861033333834105,
 'max_s': 0.007507000002078712,
 'rows': 11,
 'result': [('transit', 4588, 4.23, 112.95),
  ('radial velocity', 841, 8.88, 967.02),
  ('microlensing', 268, 10.04, 835.53),
  ('imaging', 76, 15.32, 4492.27),
  ('transit timing variations', 40, 6.46, 483.47),
  ('eclipse timing variations', 14, 12.91, 2092.93),
  ('astrometry', 6, 12.45, 4673.61),
  ('orbital brightness modulation', 6, 9.65, 350.32),
  ('pulsar timing', 2, 7.56, 127.31),
  ('disk kinematics', 1, 13.3, 794.58),
  ('pulsation timing variations', 1, 12.4, 3750.39)]}

### query critica 2 - baseline

In [19]:
con.execute("DROP TABLE IF EXISTS dim_host_w11")

con.execute("""
CREATE TABLE dim_host_w11 AS
SELECT
  hostname_canon,
  MAX(sy_dist) AS sy_dist,
  MAX(st_teff) AS st_teff,
  MAX(st_mass) AS st_mass
FROM silver_planet_v3
GROUP BY hostname_canon
""")

q2_baseline = """
WITH joined AS (
  SELECT *
  FROM silver_planet_v3 p
  JOIN dim_host_w11 h
    ON p.hostname_canon = h.hostname_canon
)
SELECT
  hostname_canon,
  COUNT(*) AS n_planets,
  ROUND(AVG(pl_rade), 2) AS avg_radius,
  ROUND(AVG(st_teff), 2) AS avg_st_teff
FROM joined
WHERE pl_rade IS NOT NULL
GROUP BY hostname_canon
ORDER BY n_planets DESC
LIMIT 15
"""

q2_base_time = timed_query(q2_baseline)
q2_base_time

{'min_s': 0.009554299998853821,
 'avg_s': 0.011644833333169421,
 'max_s': 0.012771299996529706,
 'rows': 15,
 'result': [('koi-351', 8, 3.9, 6059.66),
  ('trappist-1', 7, 0.98, 2566.0),
  ('kepler-80', 6, 1.81, 4540.0),
  ('hd 191939', 6, 6.59, 5348.0),
  ('kepler-20', 6, 2.29, 5495.0),
  ('k2-138', 6, 2.58, 5356.3),
  ('hd 110067', 6, 2.43, 5266.0),
  ('hd 34445', 6, 8.86, 5843.17),
  ('toi-1136', 6, 3.05, 5770.0),
  ('kepler-11', 6, 2.97, 5663.0),
  ('hip 41378', 6, 4.18, 6329.83),
  ('hd 219134', 6, 3.67, 4770.33),
  ('toi-178', 6, 2.22, 4316.0),
  ('kepler-82', 5, 3.7, 5411.8),
  ('k2-268', 5, 1.85, 5106.0)]}

guardando el ` explain analize`

In [20]:
explain_q2 = con.sql("EXPLAIN ANALYZE " + q2_baseline).fetchall()
explain_q2_text = "\n".join(r[1] for r in explain_q2)

out_q2 = ART_DIR / "w11_explain_critical_q2.txt"
out_q2.write_text(explain_q2_text, encoding="utf-8")

print(explain_q2_text)
print("Guardado en:", out_q2)

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
EXPLAIN ANALYZE  WITH joined AS (   SELECT *   FROM silver_planet_v3 p   JOIN dim_host_w11 h     ON p.hostname_canon = h.hostname_canon ) SELECT   hostname_canon,   COUNT(*) AS n_planets,   ROUND(AVG(pl_rade), 2) AS avg_radius,   ROUND(AVG(st_teff), 2) AS avg_st_teff FROM joined WHERE pl_rade IS NOT NULL GROUP BY hostname_canon ORDER BY n_planets DESC LIMIT 15 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0141s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│

### reescritura query critica 2

In [21]:
q2_rewrite = """
SELECT
  p.hostname_canon,
  COUNT(*) AS n_planets,
  ROUND(AVG(p.pl_rade), 2) AS avg_radius,
  ROUND(AVG(h.st_teff), 2) AS avg_st_teff
FROM (
  SELECT hostname_canon, pl_rade
  FROM silver_planet_v3
  WHERE pl_rade IS NOT NULL
) p
JOIN (
  SELECT hostname_canon, st_teff
  FROM dim_host_w11
  WHERE st_teff IS NOT NULL
) h
  ON p.hostname_canon = h.hostname_canon
GROUP BY p.hostname_canon
ORDER BY n_planets DESC
LIMIT 15
"""

q2_rewrite_time = timed_query(q2_rewrite)
q2_rewrite_time

{'min_s': 0.007786699999996927,
 'avg_s': 0.009180666667816695,
 'max_s': 0.011273000003711786,
 'rows': 15,
 'result': [('koi-351', 8, 3.9, 6080.0),
  ('trappist-1', 7, 0.98, 2566.0),
  ('hd 219134', 6, 3.67, 4913.0),
  ('toi-178', 6, 2.22, 4316.0),
  ('hip 41378', 6, 4.18, 6371.0),
  ('kepler-11', 6, 2.97, 5663.0),
  ('kepler-20', 6, 2.29, 5495.0),
  ('hd 34445', 6, 8.86, 5879.0),
  ('k2-138', 6, 2.58, 5356.3),
  ('kepler-80', 6, 1.81, 4540.0),
  ('toi-1136', 6, 3.05, 5770.0),
  ('hd 191939', 6, 6.59, 5348.0),
  ('hd 110067', 6, 2.43, 5266.0),
  ('kepler-444', 5, 0.54, 5046.0),
  ('kepler-238', 5, 2.96, 5800.0)]}

## comparación

## gold mart propuesto

In [25]:
con.execute("DROP VIEW IF EXISTS gold_perf_method_era")

con.execute("""
CREATE VIEW gold_perf_method_era AS
SELECT
  discoverymethod_canon,
  CASE
    WHEN disc_year_int IS NULL THEN 'unknown'
    WHEN disc_year_int < 2000 THEN 'pre_2000'
    WHEN disc_year_int BETWEEN 2000 AND 2009 THEN '2000s'
    WHEN disc_year_int BETWEEN 2010 AND 2019 THEN '2010s'
    WHEN disc_year_int >= 2020 THEN '2020s'
    ELSE 'unknown'
  END AS disc_era,
  COUNT(*) AS n_planets,
  ROUND(AVG(pl_rade), 2) AS avg_radius,
  ROUND(AVG(pl_bmasse), 2) AS avg_mass
FROM silver_planet_v3
WHERE discoverymethod_canon IS NOT NULL
GROUP BY discoverymethod_canon, disc_era
""")

con.sql("""
SELECT *
FROM gold_perf_method_era
ORDER BY n_planets DESC
LIMIT 15
""").show()

┌───────────────────────────┬──────────┬───────────┬────────────┬──────────┐
│   discoverymethod_canon   │ disc_era │ n_planets │ avg_radius │ avg_mass │
│          varchar          │ varchar  │   int64   │   double   │  double  │
├───────────────────────────┼──────────┼───────────┼────────────┼──────────┤
│ transit                   │ 2010s    │      3065 │       3.92 │     89.5 │
│ transit                   │ 2020s    │      1524 │       4.84 │   159.88 │
│ radial velocity           │ 2010s    │       454 │       9.72 │   941.24 │
│ radial velocity           │ 2020s    │       411 │       7.99 │  1056.45 │
│ radial velocity           │ 2000s    │       289 │      12.09 │  1113.39 │
│ microlensing              │ 2020s    │       188 │       9.78 │   837.63 │
│ microlensing              │ 2010s    │        80 │      10.63 │   830.58 │
│ transit                   │ 2000s    │        61 │      13.53 │   710.64 │
│ imaging                   │ 2020s    │        43 │      14.97 │  4364.47 │

### validación

In [24]:
con.sql("""
SELECT
  SUM(n_planets) AS n_gold_rows
FROM gold_perf_method_era
""").show()

con.sql("""
SELECT
  COUNT(*) AS n_source_rows
FROM silver_planet_v3
WHERE discoverymethod_canon IS NOT NULL
""").show()

┌─────────────┐
│ n_gold_rows │
│   int128    │
├─────────────┤
│        6291 │
└─────────────┘

┌───────────────┐
│ n_source_rows │
│     int64     │
├───────────────┤
│          6291 │
└───────────────┘

